# Simple LLM Workflow — LangGraph

A simple workflow to understand how an LLM call fits into LangGraph's
**State**, **Node**, and **Graph** pattern.

**What it does:**

- Takes ONE thing as INPUT: a `question` from the user.
- Sends that question to an LLM to get an answer.
- Returns the LLM's answer as the OUTPUT.

**How it's built:**

- The whole workflow is represented as a **GRAPH** — even though there's
  just one LLM-calling step here, it's still built the LangGraph way
  (node + edges + state) so the same pattern can be extended later
  (e.g. chaining more steps, adding routing, etc.).
- It has a **STATE** — a shared dictionary that holds `question` and
  `answer`. The node reads `question` from the state, calls the LLM,
  and writes the result back into the state as `answer`.


In [1]:
from typing import TypedDict


class LLMState(TypedDict):
    question: str
    answer: str

In [2]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(model="llama-3.1-8b-instant")

/Users/apekshagangurde/Desktop/Langchain models/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def llm_qa(state: LLMState) -> LLMState:
    question = state["question"]

    # form a prompt
    prompt = f"Answer the following question:\n{question}"

    # ask the question to the llm
    answer = model.invoke(prompt).content

    # update the answer in the state
    state["answer"] = answer

    return state

In [4]:
from langgraph.graph import StateGraph, START, END

# define the graph
graph = StateGraph(LLMState)

# define nodes
graph.add_node("llm_qa", llm_qa)

# add edges
graph.add_edge(START, "llm_qa")
graph.add_edge("llm_qa", END)

# compile graph
workflow = graph.compile()

In [5]:
# execute the graph
initial_state = {"question": "What is the capital of France?"}
final_state = workflow.invoke(initial_state)
print(final_state)

{'question': 'What is the capital of France?', 'answer': 'The capital of France is Paris.'}
